In [1]:
pip install haystack-ai

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install datasets>=2.6.1

Note: you may need to restart the kernel to use updated packages.


In [3]:
from haystack import Document,Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder,SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator

In [4]:
import datasets
from datasets import load_dataset
document_store= InMemoryDocumentStore()
dataset = load_dataset("bilgeyucel/seven-wonders", split="train")

In [5]:
docs = [Document(doc["content"],meta = doc["meta"]) for doc in dataset]

In [6]:
doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()

In [7]:
doc_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

151

In [8]:
text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
retreiver = InMemoryEmbeddingRetriever(document_store = document_store)

In [9]:
from haystack.dataclasses import ChatMessage

template =[ ChatMessage.from_user(
     """
Given the following information, answer the question.

Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}

Question: {{question}}
Answer:
"""
    
)]
prompt_builder = ChatPromptBuilder(template=template)

In [10]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
api_key=os.getenv('OPENAI_API_KEY')
generator = OpenAIChatGenerator(model = "gpt-4o-mini")

In [11]:
qa_rag_pipeline = Pipeline()
qa_rag_pipeline.add_component("text_embedder",text_embedder)
qa_rag_pipeline.add_component("retreiver",retreiver)
qa_rag_pipeline.add_component("prompt_builder",prompt_builder)
qa_rag_pipeline.add_component("generator",generator)




In [12]:
qa_rag_pipeline.connect("text_embedder.embedding","retreiver")
qa_rag_pipeline.connect("retreiver","prompt_builder")
qa_rag_pipeline.connect("prompt_builder.prompt","generator.messages")



🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retreiver: InMemoryEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - generator: OpenAIChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retreiver.query_embedding (List[float])
  - retreiver.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> generator.messages (List[ChatMessage])

In [13]:
question = "What does weather in LA?"
response = qa_rag_pipeline.run({"text_embedder":{"text":question},"prompt_builder": {"question": question}})

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
print(response["generator"]["replies"][0].text)

The question seems to be phrased incorrectly. If you meant to ask about the weather in Los Angeles (LA), I can tell you that LA typically has a Mediterranean climate, characterized by warm, dry summers and mild, wet winters. Summers can see temperatures well into the 80s and 90s Fahrenheit, while winters usually remain in the 50s to 70s. Rain is less common, primarily falling between November and March. If you have a specific aspect of LA's weather you're curious about, please clarify!
